# CLIP-ViT + LoRA Continual Learning on Split CIFAR-100

This notebook follows the simplified supervisor-aligned setup.

Main setup:

- CLIP-ViT vision encoder: `openai/clip-vit-base-patch16`
- Split CIFAR-100 continual learning
- 5 steps, 20 classes per step
- Debug first with 1 epoch
- Keep the comparison simple and explainable

Important:

- This is **CLIP-ViT**, not standard supervised ImageNet ViT.
- The CLIP text encoder is not used.
- Only the CLIP vision encoder is used.
- A new CIFAR-100 classifier is trained on top of the CLIP vision representation.
- For CLIP-ViT, LoRA target modules are `q_proj` and `v_proj`.

Main comparison:

1. `seq_ft_no_replay`
2. `simple_avg_no_replay`
3. `simple_avg_replay`
4. `do_merging_simple`
5. `orthogonal_loss`
6. `rank_extension`
7. `joint_upper_bound`

The main method is:

`simple_avg_no_replay`

The replay version is added only beside simple average as a diagnostic:

`simple_avg_replay`

Supervisor-suggested optional methods:

- `orthogonal_loss`: sequential LoRA training where previous LoRA updates are absorbed into the model after each step, and the next LoRA is trained with the supervisor's orthogonality-style loss term.
- `rank_extension`: one LoRA whose rank grows step by step; old rank parts are copied and frozen, and only the new rank block is trained.

Paper-inspired optional method:

- `do_merging_simple`: simplified DO-Merging-inspired LoRA merge. It is not a full paper reproduction yet.


In [ ]:
# ============================================================
# 1) Imports
# ============================================================

import os
import gc
import json
import random
import math
import inspect
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset, concatenate_datasets
from torchvision import transforms

from transformers import (
    CLIPImageProcessor,
    CLIPVisionModel,
    TrainingArguments,
    Trainer,
    set_seed,
)

from transformers.modeling_outputs import ImageClassifierOutput

from peft import LoraConfig, get_peft_model

# For notebook and .py compatibility
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

In [ ]:
# ============================================================
# 2) Full comparison configuration
# ============================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEBUG_MODE = False

RUN_NAME = "clip_vit_lora_cifar100_full_comparison_with_do_exact_style"

MODEL_CHECKPOINT = "openai/clip-vit-base-patch16"

NUM_CLASSES = 100
NUM_STEPS = 5
CLASSES_PER_STEP = 20

# ============================================================
# Epoch setup
# ============================================================

# FT_EPOCHS = 5
# LORA_EPOCHS = 5
# JOINT_EPOCHS = 5
# RANKEXT_EPOCHS = 5

# If too heavy:
FT_EPOCHS = 3
LORA_EPOCHS = 3
JOINT_EPOCHS = 3
RANKEXT_EPOCHS = 3

# ============================================================
# Batch settings
# ============================================================

BATCH_FT = 8
ACCUM_FT = 2

BATCH_LORA = 16
ACCUM_LORA = 1

# ============================================================
# Learning rates
# ============================================================

LR_FT = 3e-5
LR_LORA = 5e-5
LR_JOINT = 5e-5
LR_RANKEXT = 1e-4

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

# ============================================================
# LoRA setup for CLIP-ViT
# ============================================================

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# ============================================================
# Replay
# ============================================================

REPLAY_PER_CLASS = 20

# ============================================================
# Orthogonal regularization on selected methods
# ============================================================

SIMPLE_AVG_ORTH_LAMBDA = 1e-3
RANKEXT_ORTH_LAMBDA = 1e-3

# ============================================================
# Rank-extension setup
# ============================================================

RANKEXT_NEW_RANK_PER_STEP = 8
RANKEXT_ALPHA_PER_RANK = 2.0

# ============================================================
# DO-Merging-style data-free orthogonalization setup
# ============================================================
# This is the closer paper-style variant.
#
# It does:
# 1. Data-free orthogonalization of LoRA A/B components across tasks.
# 2. Build full delta_W = B @ A.
# 3. Column-wise decouple delta_W into magnitude and direction.
# 4. Merge magnitude and direction separately.

DO_ORTH_STEPS = 100
DO_ORTH_LR = 0.05
DO_PRESERVE_WEIGHT = 1.0
DO_ORTH_WEIGHT = 1.0

# If unstable or too slow:
# DO_ORTH_STEPS = 50
# DO_ORTH_LR = 0.02

# ============================================================
# Full comparison
# ============================================================

METHODS_TO_RUN = {
    "seq_ft_no_replay": True,
    "simple_avg_no_replay": True,
    "simple_avg_replay": True,
    "simple_avg_replay_orth": True,
    "do_merging_simple": True,
    "do_merging_datafree_orth": True,
    "rank_extension": True,
    "rank_extension_orth": True,
    "joint_upper_bound": True,
}

# ============================================================
# Fixed output path on cluster
# ============================================================

ROOT_RESULTS_DIR = "/nfsd/lttm4/tesisti/shahrampour/projects/runs/cifar100_5step_n5_main/results"

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")

BASE_OUTPUT_DIR = os.path.join(
    ROOT_RESULTS_DIR,
    f"{RUN_NAME}_{RUN_TAG}"
)

TABLES_DIR = os.path.join(BASE_OUTPUT_DIR, "tables")
PLOTS_DIR = os.path.join(BASE_OUTPUT_DIR, "plots")
MODELS_DIR = os.path.join(BASE_OUTPUT_DIR, "models")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

all_results = []

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("FP16:", USE_FP16)
print("Checkpoint:", MODEL_CHECKPOINT)
print("BASE_OUTPUT_DIR:", BASE_OUTPUT_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("PLOTS_DIR:", PLOTS_DIR)
print("MODELS_DIR:", MODELS_DIR)

print("\nEpochs:")
print({
    "FT_EPOCHS": FT_EPOCHS,
    "LORA_EPOCHS": LORA_EPOCHS,
    "JOINT_EPOCHS": JOINT_EPOCHS,
    "RANKEXT_EPOCHS": RANKEXT_EPOCHS,
})

print("\nLoRA:")
print({
    "LORA_R": LORA_R,
    "LORA_ALPHA": LORA_ALPHA,
    "LORA_DROPOUT": LORA_DROPOUT,
    "TARGET_MODULES": TARGET_MODULES,
})

print("\nDO-Merging-style:")
print({
    "DO_ORTH_STEPS": DO_ORTH_STEPS,
    "DO_ORTH_LR": DO_ORTH_LR,
    "DO_PRESERVE_WEIGHT": DO_PRESERVE_WEIGHT,
    "DO_ORTH_WEIGHT": DO_ORTH_WEIGHT,
})

print("\nRank extension:")
print({
    "RANKEXT_NEW_RANK_PER_STEP": RANKEXT_NEW_RANK_PER_STEP,
    "RANKEXT_ALPHA_PER_RANK": RANKEXT_ALPHA_PER_RANK,
    "LR_RANKEXT": LR_RANKEXT,
})

print("\nMethods:")
print(json.dumps(METHODS_TO_RUN, indent=2))

In [ ]:
# ============================================================
# 3) Load CIFAR-100 and define continual splits
# ============================================================

dataset = load_dataset("cifar100")

# Hugging Face CIFAR-100 usually has columns:
# img, fine_label, coarse_label
LABEL_COL = "fine_label" if "fine_label" in dataset["train"].column_names else "label"
IMAGE_COL = "img" if "img" in dataset["train"].column_names else "image"

class_splits = [
    list(range(0, 20)),
    list(range(20, 40)),
    list(range(40, 60)),
    list(range(60, 80)),
    list(range(80, 100)),
]

first_step_classes = class_splits[0]
later_step_classes = [c for split in class_splits[1:] for c in split]
all_classes = [c for split in class_splits for c in split]

def classes_for_step(step_idx):
    return class_splits[step_idx]

def filter_by_classes(ds, class_ids):
    class_ids = set(class_ids)
    return ds.filter(lambda x: int(x[LABEL_COL]) in class_ids)

print("Dataset columns:", dataset["train"].column_names)
print("Label column:", LABEL_COL)
print("Image column:", IMAGE_COL)
for i, cls in enumerate(class_splits, start=1):
    print(f"Step {i}: {cls[0]}-{cls[-1]}")

In [ ]:
# ============================================================
# 4) CLIP image processor and transforms
# ============================================================

image_processor = CLIPImageProcessor.from_pretrained(MODEL_CHECKPOINT)

if hasattr(image_processor, "crop_size") and image_processor.crop_size is not None:
    H = int(image_processor.crop_size.get("height", 224))
    W = int(image_processor.crop_size.get("width", 224))
else:
    H = W = 224

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.05,
        contrast=0.05,
        saturation=0.05,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = np.squeeze(x).astype(np.uint8)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {
        "pixel_values": pixel_values,
        "labels": labels,
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": float((preds == labels).mean())
    }

print("Image size:", H, W)
print("CLIP mean:", image_processor.image_mean)
print("CLIP std:", image_processor.image_std)

In [ ]:
# ============================================================
# 5) CLIP-ViT vision classifier wrapper
# ============================================================

class CLIPVisionForCIFAR100(nn.Module):
    """
    CLIP-ViT vision encoder + trainable CIFAR-100 classifier.

    This uses:
    openai/clip-vit-base-patch16

    The text encoder is not used.
    Only the CLIP vision backbone is used.
    """

    def __init__(self, checkpoint, num_labels):
        super().__init__()

        self.vision_model = CLIPVisionModel.from_pretrained(checkpoint)

        hidden_size = self.vision_model.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

        self.config = self.vision_model.config
        self.config.num_labels = num_labels
        self.config.id2label = {i: str(i) for i in range(num_labels)}
        self.config.label2id = {str(i): i for i in range(num_labels)}

    def forward(
        self,
        pixel_values=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=True,
        **kwargs,
    ):
        outputs = self.vision_model(
            pixel_values=pixel_values,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        pooled_output = outputs.pooler_output
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return ImageClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

def fresh_pretrained_model():
    """
    Fresh CLIP-ViT vision model with a CIFAR-100 classifier.
    """
    return CLIPVisionForCIFAR100(
        checkpoint=MODEL_CHECKPOINT,
        num_labels=NUM_CLASSES,
    )

def add_lora(model):
    """
    Add LoRA to CLIP-ViT attention q_proj and v_proj modules.
    """
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        modules_to_save=["classifier"],
    )

    model = get_peft_model(model, lora_config)
    return model

print("LoRA target modules:", TARGET_MODULES)

In [ ]:
# ============================================================
# 6) Dataset helpers
# ============================================================

def build_replay_dataset(old_classes, replay_per_class):
    if len(old_classes) == 0 or replay_per_class <= 0:
        return None

    parts = []

    for cls in old_classes:
        cls_ds = filter_by_classes(dataset["train"], [cls])
        n = min(replay_per_class, len(cls_ds))
        cls_ds = cls_ds.shuffle(seed=SEED).select(range(n))
        parts.append(cls_ds)

    replay_ds = concatenate_datasets(parts)
    return replay_ds

def make_train_dataset(step_idx, replay_per_class=0):
    current_classes = classes_for_step(step_idx)
    current_ds = filter_by_classes(dataset["train"], current_classes)

    old_classes = []
    for old_step in range(step_idx):
        old_classes.extend(classes_for_step(old_step))

    replay_ds = build_replay_dataset(
        old_classes=old_classes,
        replay_per_class=replay_per_class,
    )

    if replay_ds is None:
        final_ds = current_ds
    else:
        final_ds = concatenate_datasets([current_ds, replay_ds])

    final_ds = final_ds.shuffle(seed=SEED + step_idx)
    final_ds = final_ds.with_transform(preprocess_train)

    print(
        f"Step {step_idx + 1} | "
        f"current={len(current_ds)} | "
        f"replay={0 if replay_ds is None else len(replay_ds)} | "
        f"total={len(final_ds)}"
    )

    return final_ds

def make_eval_dataset(class_ids):
    ds = filter_by_classes(dataset["test"], class_ids)
    ds = ds.with_transform(preprocess_val)
    return ds

def make_joint_train_dataset():
    ds = dataset["train"].shuffle(seed=SEED)
    ds = ds.with_transform(preprocess_train)
    return ds

def make_joint_eval_dataset():
    ds = dataset["test"]
    ds = ds.with_transform(preprocess_val)
    return ds

eval_first = make_eval_dataset(first_step_classes)
eval_later = make_eval_dataset(later_step_classes)
eval_all_seen = make_eval_dataset(all_classes)

print("first_step eval:", len(eval_first))
print("later_steps eval:", len(eval_later))
print("all_seen eval:", len(eval_all_seen))

In [ ]:
# ============================================================
# 7) Trainer helpers
# ============================================================

def get_training_args(
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    train_dataset_len=None,
    eval_strategy="epoch",
):
    """
    Trainer settings.

    We use warmup_steps instead of warmup_ratio because warmup_ratio is deprecated
    in newer Transformers versions.
    """

    if train_dataset_len is not None:
        steps_per_epoch = math.ceil(train_dataset_len / batch_size / accum_steps)
        total_steps = int(steps_per_epoch * epochs)
        warmup_steps = int(WARMUP_RATIO * total_steps)
    else:
        warmup_steps = 0

    kwargs = dict(
        output_dir=output_dir,
        remove_unused_columns=False,
        save_strategy="no",
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=accum_steps,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        report_to="none",
        max_grad_norm=1.0,
    )

    sig = inspect.signature(TrainingArguments.__init__)

    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = eval_strategy
    else:
        kwargs["evaluation_strategy"] = eval_strategy

    return TrainingArguments(**kwargs)


def train_with_trainer(
    model,
    train_ds,
    eval_ds,
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    trainer_cls=Trainer,
    **trainer_kwargs,
):
    args = get_training_args(
        output_dir=output_dir,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        accum_steps=accum_steps,
        train_dataset_len=len(train_ds),
        eval_strategy="epoch",
    )

    trainer = trainer_cls(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        **trainer_kwargs,
    )

    trainer.train()
    eval_out = trainer.evaluate()

    return trainer, eval_out


def evaluate_model(model, method_name):
    args = get_training_args(
        output_dir=os.path.join(MODELS_DIR, f"eval_{method_name}"),
        epochs=1,
        lr=LR_LORA,
        batch_size=BATCH_LORA,
        accum_steps=ACCUM_LORA,
        train_dataset_len=None,
        eval_strategy="no",
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    eval_first_out = trainer.evaluate(eval_dataset=eval_first)
    eval_later_out = trainer.evaluate(eval_dataset=eval_later)
    eval_all_out = trainer.evaluate(eval_dataset=eval_all_seen)

    rows = [
        {
            "method": method_name,
            "eval_set": "first_step",
            "accuracy": float(eval_first_out["eval_accuracy"]),
            "loss": float(eval_first_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "later_steps",
            "accuracy": float(eval_later_out["eval_accuracy"]),
            "loss": float(eval_later_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "all_seen",
            "accuracy": float(eval_all_out["eval_accuracy"]),
            "loss": float(eval_all_out["eval_loss"]),
        },
    ]

    all_results.extend(rows)

    print(pd.DataFrame(rows))
    return rows

In [ ]:
# ============================================================
# 8) LoRA extraction and merge helpers
# ============================================================

def normalize_module_name(name):
    prefixes = [
        "base_model.model.",
        "model.",
    ]

    out = name

    for p in prefixes:
        if out.startswith(p):
            out = out[len(p):]

    return out


def extract_lora_state(model):
    """
    Extract:
    - full LoRA delta_W per target module
    - original low-rank A/B LoRA components
    - classifier weights

    This is needed for:
    - simple average
    - simple average replay
    - DO-Merging-style A/B orthogonalization
    """

    state = {
        "deltas": {},
        "lora_ab": {},
        "classifier_weight": None,
        "classifier_bias": None,
    }

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        A = module.lora_A["default"].weight.detach().cpu().float()
        B = module.lora_B["default"].weight.detach().cpu().float()

        scaling = (
            module.scaling["default"]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        scaling = float(scaling)

        delta = (B @ A) * scaling

        plain_name = normalize_module_name(name)

        state["deltas"][plain_name] = delta

        state["lora_ab"][plain_name] = {
            "A": A,
            "B": B,
            "scaling": scaling,
            "rank": int(A.shape[0]),
            "in_features": int(A.shape[1]),
            "out_features": int(B.shape[0]),
        }

    # Extract classifier saved by PEFT modules_to_save
    for name, tensor in model.state_dict().items():
        if "classifier.modules_to_save.default.weight" in name:
            state["classifier_weight"] = tensor.detach().cpu().clone()

        if "classifier.modules_to_save.default.bias" in name:
            state["classifier_bias"] = tensor.detach().cpu().clone()

    return state


def get_submodule_by_name(model, module_name):
    module_name = normalize_module_name(module_name)

    current = model

    for part in module_name.split("."):
        if part == "":
            continue
        current = getattr(current, part)

    return current


def simple_average_deltas(step_states):
    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}

    for key in keys:
        vals = []

        for state in step_states:
            if key in state["deltas"]:
                vals.append(state["deltas"][key].float())

        merged[key] = torch.stack(vals, dim=0).mean(dim=0)

    return merged


def do_merge_deltas(step_states):
    """
    Simplified DO-inspired merge.

    This is NOT the full paper-style DO-Merging.
    It only decouples each full delta into one global magnitude and direction.
    """

    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}

    eps = 1e-8

    for key in keys:
        deltas = []

        for state in step_states:
            if key in state["deltas"]:
                deltas.append(state["deltas"][key].float())

        norms = [torch.linalg.norm(d).clamp_min(eps) for d in deltas]
        directions = [d / n for d, n in zip(deltas, norms)]

        mean_norm = torch.stack(norms).mean()
        mean_direction = torch.stack(directions, dim=0).mean(dim=0)
        mean_direction = mean_direction / torch.linalg.norm(mean_direction).clamp_min(eps)

        merged[key] = mean_norm * mean_direction

    return merged


def apply_deltas_to_base(merged_deltas, step_states):
    """
    Apply merged LoRA deltas to a fresh CLIP-ViT model and stitch classifier rows.
    """

    model = fresh_pretrained_model()

    with torch.no_grad():
        for key, delta in merged_deltas.items():
            try:
                module = get_submodule_by_name(model, key)
            except Exception as e:
                print("Could not find module:", key, "|", e)
                continue

            if not hasattr(module, "weight"):
                print("Module has no weight:", key)
                continue

            module.weight.add_(
                delta.to(
                    device=module.weight.device,
                    dtype=module.weight.dtype,
                )
            )

        # classifier stitching
        for step_idx, state in enumerate(step_states):
            classes = classes_for_step(step_idx)

            if state["classifier_weight"] is None:
                continue

            w = state["classifier_weight"].to(model.classifier.weight.device)
            b = state["classifier_bias"].to(model.classifier.bias.device)

            for c in classes:
                model.classifier.weight[c].copy_(w[c])
                model.classifier.bias[c].copy_(b[c])

    return model


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 9) Baseline: seq_ft_no_replay
# ============================================================

if METHODS_TO_RUN["seq_ft_no_replay"]:
    seq_ft_model = fresh_pretrained_model()

    for step_idx in range(NUM_STEPS):
        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"seq_ft_no_replay_step_{step_idx + 1}",
        )

        print(
            f"\n===== seq_ft_no_replay | "
            f"step {step_idx + 1}/{NUM_STEPS} ====="
        )

        train_with_trainer(
            model=seq_ft_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=FT_EPOCHS,
            lr=LR_FT,
            batch_size=BATCH_FT,
            accum_steps=ACCUM_FT,
        )

    evaluate_model(seq_ft_model, "seq_ft_no_replay")

    del seq_ft_model
    cleanup()

else:
    print("Skipping seq_ft_no_replay")

In [ ]:
# ============================================================
# 10) Train independent LoRAs
# ============================================================

def train_independent_loras(
    method_prefix,
    replay_per_class=0,
    trainer_cls=Trainer,
    trainer_extra_kwargs_fn=None,
    epochs=LORA_EPOCHS,
    lr=LR_LORA,
):
    """
    Train one independent LoRA per step.

    Used for:
    - simple_avg_no_replay
    - simple_avg_replay
    - simple_avg_replay_orth
    - do_merging_simple

    If trainer_cls is a custom Trainer, it can add regularization terms.
    """

    step_states = []

    for step_idx in range(NUM_STEPS):
        model = fresh_pretrained_model()
        model = add_lora(model)

        model.print_trainable_parameters()

        train_ds = make_train_dataset(
            step_idx=step_idx,
            replay_per_class=replay_per_class,
        )

        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"{method_prefix}_step_{step_idx + 1}",
        )

        extra_kwargs = {}
        if trainer_extra_kwargs_fn is not None:
            extra_kwargs = trainer_extra_kwargs_fn(step_idx, step_states)

        print(
            f"\n===== {method_prefix} | "
            f"step {step_idx + 1}/{NUM_STEPS} | "
            f"replay_per_class={replay_per_class} ====="
        )

        train_with_trainer(
            model=model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=epochs,
            lr=lr,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
            trainer_cls=trainer_cls,
            **extra_kwargs,
        )

        state = extract_lora_state(model)
        step_states.append(state)

        del model
        cleanup()

    return step_states

In [ ]:
# ============================================================
# 11) Simple average without replay
# ============================================================

step_states_no_replay = None

needs_no_replay_sources = (
    METHODS_TO_RUN.get("simple_avg_no_replay", False)
    or METHODS_TO_RUN.get("do_merging_simple", False)
    or METHODS_TO_RUN.get("do_merging_datafree_orth", False)
)

if needs_no_replay_sources:
    step_states_no_replay = train_independent_loras(
        method_prefix="simple_avg_no_replay_source",
        replay_per_class=0,
    )

    if METHODS_TO_RUN.get("simple_avg_no_replay", False):
        simple_delta = simple_average_deltas(step_states_no_replay)

        simple_model = apply_deltas_to_base(
            merged_deltas=simple_delta,
            step_states=step_states_no_replay,
        )

        evaluate_model(simple_model, "simple_avg_no_replay")

        del simple_model
        cleanup()

else:
    print("Skipping simple_avg_no_replay source training")

In [ ]:
# ============================================================
# 12) simple_avg_replay
# ============================================================

step_states_replay = None

if METHODS_TO_RUN["simple_avg_replay"]:
    step_states_replay = train_independent_loras(
        method_prefix="simple_avg_replay_source",
        replay_per_class=REPLAY_PER_CLASS,
    )

    replay_delta = simple_average_deltas(step_states_replay)
    replay_model = apply_deltas_to_base(
        merged_deltas=replay_delta,
        step_states=step_states_replay,
    )

    evaluate_model(replay_model, "simple_avg_replay")

    del replay_model
    cleanup()

else:
    print("Skipping simple_avg_replay")

In [ ]:
# ============================================================
# 13) DO-Merging variants
# ============================================================
# do_merging_simple:
#   Previous simplified magnitude-direction merge.
#
# do_merging_datafree_orth:
#   Closer paper-style DO-Merging implementation:
#   1. Data-free orthogonalization of LoRA A/B components across tasks.
#   2. Build full delta_W = B @ A.
#   3. Column-wise decouple delta_W into magnitude and direction.
#   4. Merge magnitude and direction separately.

def columnwise_decouple_delta(delta, eps=1e-8):
    """
    Decouple full-rank delta matrix column-wise.

    delta shape:
        [out_features, in_features]

    magnitude:
        ||delta[:, j]|| for each column j

    direction:
        delta[:, j] / magnitude[j]
    """

    mag = torch.linalg.norm(delta, dim=0, keepdim=True).clamp_min(eps)
    direction = delta / mag

    return mag, direction


def columnwise_recouple_delta(mag, direction):
    """
    Reconstruct delta from magnitude and direction.
    """

    return direction * mag


def merge_decoupled_deltas_columnwise(delta_list, weights=None, eps=1e-8):
    """
    Merge full deltas after column-wise decoupling.

    For each delta:
        delta = direction * magnitude

    Merge:
        merged_magnitude = weighted average of magnitudes
        merged_direction = normalized weighted average of directions
    """

    if len(delta_list) == 0:
        raise ValueError("delta_list is empty.")

    device = delta_list[0].device
    dtype = delta_list[0].dtype

    if weights is None:
        weights = torch.ones(len(delta_list), device=device, dtype=dtype)
        weights = weights / weights.sum().clamp_min(eps)
    else:
        weights = torch.tensor(weights, device=device, dtype=dtype)
        weights = weights / weights.sum().clamp_min(eps)

    mags = []
    dirs = []

    for delta in delta_list:
        mag, direction = columnwise_decouple_delta(delta, eps=eps)
        mags.append(mag)
        dirs.append(direction)

    merged_mag = torch.zeros_like(mags[0])
    merged_dir = torch.zeros_like(dirs[0])

    for w, mag, direction in zip(weights, mags, dirs):
        merged_mag = merged_mag + w * mag
        merged_dir = merged_dir + w * direction

    merged_dir_norm = torch.linalg.norm(
        merged_dir,
        dim=0,
        keepdim=True,
    ).clamp_min(eps)

    merged_dir = merged_dir / merged_dir_norm

    merged_delta = columnwise_recouple_delta(
        mag=merged_mag,
        direction=merged_dir,
    )

    return merged_delta


def pairwise_lora_component_orth_loss(params, component_type, eps=1e-8):
    """
    Orthogonal constraint between LoRA components across tasks.

    For A matrices:
        A shape = [rank, in_features]
        We reduce overlap between row subspaces using A_i @ A_j.T.

    For B matrices:
        B shape = [out_features, rank]
        We reduce overlap between column subspaces using B_i.T @ B_j.

    This is the practical low-rank component orthogonalization used before
    building full delta_W.
    """

    if len(params) <= 1:
        return torch.tensor(0.0, device=params[0].device, dtype=params[0].dtype)

    loss = torch.tensor(0.0, device=params[0].device, dtype=params[0].dtype)
    count = 0

    for i in range(len(params)):
        Xi = params[i]

        for j in range(i + 1, len(params)):
            Xj = params[j]

            if component_type == "A":
                cross = Xi @ Xj.T
            elif component_type == "B":
                cross = Xi.T @ Xj
            else:
                raise ValueError("component_type must be 'A' or 'B'")

            denom = (
                torch.linalg.norm(Xi).pow(2)
                * torch.linalg.norm(Xj).pow(2)
            ).clamp_min(eps)

            loss = loss + torch.linalg.norm(cross).pow(2) / denom
            count += 1

    return loss / max(count, 1)


def orthogonalize_lora_components_datafree(
    A_list,
    B_list,
    steps=100,
    lr=0.05,
    preserve_weight=1.0,
    orth_weight=1.0,
    eps=1e-8,
    verbose=False,
):
    """
    Data-free optimization of LoRA components for one layer/module.

    Inputs:
        A_list: A matrices from different task LoRAs.
        B_list: B matrices from different task LoRAs.

    Optimize:
        A_hat_i and B_hat_i

    Objective:
        preserve_weight * closeness_to_original
        +
        orth_weight * cross_task_orthogonality_penalty

    This is the data-free layer-wise gradient descent part.
    """

    if len(A_list) != len(B_list):
        raise ValueError("A_list and B_list must have the same length.")

    if len(A_list) <= 1:
        return (
            [A.detach().clone() for A in A_list],
            [B.detach().clone() for B in B_list],
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float32

    A_orig = [A.detach().clone().to(device=device, dtype=dtype) for A in A_list]
    B_orig = [B.detach().clone().to(device=device, dtype=dtype) for B in B_list]

    A_params = [nn.Parameter(A.clone()) for A in A_orig]
    B_params = [nn.Parameter(B.clone()) for B in B_orig]

    optimizer = torch.optim.Adam(A_params + B_params, lr=lr)

    for step in range(steps):
        optimizer.zero_grad()

        preserve_loss = torch.tensor(0.0, device=device, dtype=dtype)

        for A_p, A_o, B_p, B_o in zip(A_params, A_orig, B_params, B_orig):
            preserve_loss = preserve_loss + F.mse_loss(A_p, A_o)
            preserve_loss = preserve_loss + F.mse_loss(B_p, B_o)

        preserve_loss = preserve_loss / len(A_params)

        orth_A = pairwise_lora_component_orth_loss(
            A_params,
            component_type="A",
            eps=eps,
        )

        orth_B = pairwise_lora_component_orth_loss(
            B_params,
            component_type="B",
            eps=eps,
        )

        orth_loss = orth_A + orth_B

        loss = preserve_weight * preserve_loss + orth_weight * orth_loss

        loss.backward()
        optimizer.step()

        if verbose and (step % 20 == 0 or step == steps - 1):
            print(
                f"[DO data-free orth] step={step:03d} | "
                f"loss={float(loss.detach().cpu()):.6f} | "
                f"preserve={float(preserve_loss.detach().cpu()):.6f} | "
                f"orth_A={float(orth_A.detach().cpu()):.6f} | "
                f"orth_B={float(orth_B.detach().cpu()):.6f}"
            )

    A_out = [A.detach().cpu().clone() for A in A_params]
    B_out = [B.detach().cpu().clone() for B in B_params]

    return A_out, B_out


def do_merge_deltas_datafree_orth(
    step_states,
    weights=None,
    verbose=False,
):
    """
    DO-Merging-style data-free orthogonal merge.

    For each LoRA target module:
        1. Collect A_i, B_i across task LoRAs.
        2. Data-free orthogonalize A_i and B_i across tasks.
        3. Build delta_i = B_i @ A_i * scaling_i.
        4. Column-wise decouple delta_i into magnitude and direction.
        5. Merge magnitudes and directions separately.
    """

    if len(step_states) == 0:
        raise ValueError("step_states is empty.")

    keys = sorted(step_states[0]["lora_ab"].keys())
    merged = {}

    for key_idx, key in enumerate(keys):
        A_list = []
        B_list = []
        scaling_list = []

        valid = True

        for state in step_states:
            if key not in state["lora_ab"]:
                valid = False
                break

            A_list.append(state["lora_ab"][key]["A"])
            B_list.append(state["lora_ab"][key]["B"])
            scaling_list.append(float(state["lora_ab"][key]["scaling"]))

        if not valid:
            print(f"[DO data-free orth] Missing A/B for {key}; falling back to simple delta merge.")
            delta_list = [
                state["deltas"][key].float()
                for state in step_states
                if key in state["deltas"]
            ]

            merged[key] = merge_decoupled_deltas_columnwise(
                delta_list=delta_list,
                weights=weights,
            ).cpu()

            continue

        if verbose:
            print(
                f"[DO data-free orth] module {key_idx + 1}/{len(keys)}: {key}"
            )

        A_orth, B_orth = orthogonalize_lora_components_datafree(
            A_list=A_list,
            B_list=B_list,
            steps=DO_ORTH_STEPS,
            lr=DO_ORTH_LR,
            preserve_weight=DO_PRESERVE_WEIGHT,
            orth_weight=DO_ORTH_WEIGHT,
            verbose=False,
        )

        delta_list = []

        for A, B, scaling in zip(A_orth, B_orth, scaling_list):
            delta = (B @ A) * float(scaling)
            delta_list.append(delta.float())

        merged_delta = merge_decoupled_deltas_columnwise(
            delta_list=delta_list,
            weights=weights,
        )

        merged[key] = merged_delta.cpu()

    return merged


if METHODS_TO_RUN.get("do_merging_simple", False):
    assert step_states_no_replay is not None

    do_delta = do_merge_deltas(step_states_no_replay)

    do_model = apply_deltas_to_base(
        merged_deltas=do_delta,
        step_states=step_states_no_replay,
    )

    evaluate_model(do_model, "do_merging_simple")

    del do_model
    cleanup()

else:
    print("Skipping do_merging_simple")


if METHODS_TO_RUN.get("do_merging_datafree_orth", False):
    assert step_states_no_replay is not None

    do_orth_delta = do_merge_deltas_datafree_orth(
        step_states=step_states_no_replay,
        weights=None,
        verbose=True,
    )

    do_orth_model = apply_deltas_to_base(
        merged_deltas=do_orth_delta,
        step_states=step_states_no_replay,
    )

    evaluate_model(do_orth_model, "do_merging_datafree_orth")

    del do_orth_model
    cleanup()

else:
    print("Skipping do_merging_datafree_orth")

In [ ]:
# ============================================================
# 14) Simple average + replay + orthogonal regularization
# ============================================================

def compute_lora_orth_penalty_against_previous_loras(
    model,
    previous_states,
    eps=1e-8,
):
    """
    Orthogonal regularization for independent LoRA training.

    The current LoRA delta_W is encouraged to be less aligned with
    previous trained LoRA deltas.

    This is useful for simple_avg_replay_orth because the goal is to train
    LoRAs that are more compatible before simple-average merging.

    Penalty:
        mean cosine(delta_current, delta_previous)^2
    """

    if previous_states is None or len(previous_states) == 0:
        return torch.tensor(0.0, device=next(model.parameters()).device)

    penalties = []

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        plain_name = normalize_module_name(name)

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight

        scaling = (
            module.scaling["default"]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        current_delta = (B @ A) * float(scaling)
        current_vec = current_delta.flatten()
        current_norm = torch.linalg.norm(current_vec).clamp_min(eps)

        for prev_state in previous_states:
            if plain_name not in prev_state["deltas"]:
                continue

            old_delta = prev_state["deltas"][plain_name].to(
                device=current_delta.device,
                dtype=current_delta.dtype,
            )

            old_vec = old_delta.flatten()
            old_norm = torch.linalg.norm(old_vec).clamp_min(eps)

            cosine = torch.dot(current_vec, old_vec) / (current_norm * old_norm)

            penalties.append(cosine.pow(2))

    if len(penalties) == 0:
        return torch.tensor(0.0, device=next(model.parameters()).device)

    return torch.stack(penalties).mean()


class LoRAOrthRegularizedTrainer(Trainer):
    """
    Trainer for simple_avg_replay_orth.

    Loss:
        CE loss + lambda * orthogonal penalty

    This is not a standalone method.
    It is a regularized version of independent LoRA training before merging.
    """

    def __init__(
        self,
        *args,
        previous_states=None,
        lambda_orth=1e-3,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.previous_states = previous_states or []
        self.lambda_orth = float(lambda_orth)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        outputs = model(**inputs)

        ce_loss = outputs.loss

        orth_loss = compute_lora_orth_penalty_against_previous_loras(
            model=model,
            previous_states=self.previous_states,
        )

        loss = ce_loss + self.lambda_orth * orth_loss

        return (loss, outputs) if return_outputs else loss


def simple_avg_replay_orth_extra_kwargs(step_idx, previous_states):
    """
    Extra kwargs passed to LoRAOrthRegularizedTrainer.

    Step 1 has no previous states, so orth penalty is zero.
    Step 2+ are regularized against previous LoRA deltas.
    """

    return {
        "previous_states": previous_states,
        "lambda_orth": SIMPLE_AVG_ORTH_LAMBDA,
    }


step_states_replay_orth = None

if METHODS_TO_RUN["simple_avg_replay_orth"]:
    step_states_replay_orth = train_independent_loras(
        method_prefix="simple_avg_replay_orth_source",
        replay_per_class=REPLAY_PER_CLASS,
        trainer_cls=LoRAOrthRegularizedTrainer,
        trainer_extra_kwargs_fn=simple_avg_replay_orth_extra_kwargs,
        epochs=LORA_EPOCHS,
        lr=LR_LORA,
    )

    replay_orth_delta = simple_average_deltas(step_states_replay_orth)

    replay_orth_model = apply_deltas_to_base(
        merged_deltas=replay_orth_delta,
        step_states=step_states_replay_orth,
    )

    evaluate_model(replay_orth_model, "simple_avg_replay_orth")

    del replay_orth_model
    cleanup()

else:
    print("Skipping simple_avg_replay_orth")

In [ ]:
# ============================================================
# 15) Rank extension and rank extension + orthogonal regularization
# ============================================================

class GrowingRankLoRALinear(nn.Module):
    """
    Custom growing-rank LoRA layer.

    Corrected rank-extension idea:

    Step 1:
        rank = 8

    Step 2:
        rank = 16
        first 8 rank dimensions copied from Step 1 and frozen
        new 8 rank dimensions trained

    Step 3:
        rank = 24
        first 16 rank dimensions copied from Step 2 and frozen
        new 8 rank dimensions trained

    and so on.

    The base CLIP linear layer is frozen.
    Only the new rank block is trainable during training steps.

    For final evaluation:
        new_rank can be 0.
        In that case, all rank blocks are frozen/copied and no new block is created.

    Scaling fix:
        alpha_t = RANKEXT_ALPHA_PER_RANK * total_rank
        scaling = alpha_t / total_rank = RANKEXT_ALPHA_PER_RANK

    So LoRA strength stays stable as rank grows.
    """

    def __init__(
        self,
        base_layer,
        total_rank,
        frozen_A=None,
        frozen_B=None,
        dropout=0.0,
    ):
        super().__init__()

        self.base_layer = base_layer
        self.total_rank = int(total_rank)

        if frozen_A is None or frozen_B is None:
            self.frozen_rank = 0
        else:
            self.frozen_rank = int(frozen_A.shape[0])

        self.new_rank = self.total_rank - self.frozen_rank

        if self.new_rank < 0:
            raise ValueError(
                f"new_rank cannot be negative. "
                f"total_rank={self.total_rank}, frozen_rank={self.frozen_rank}"
            )

        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features

        # Freeze original CLIP linear layer
        for p in self.base_layer.parameters():
            p.requires_grad = False

        # Rank-dependent alpha, stable scaling
        self.rankext_alpha = RANKEXT_ALPHA_PER_RANK * self.total_rank
        self.scaling = self.rankext_alpha / self.total_rank

        self.dropout = nn.Dropout(dropout)

        # Frozen old rank block
        if self.frozen_rank > 0:
            self.A_frozen = nn.Parameter(
                frozen_A.detach().clone().float(),
                requires_grad=False,
            )

            self.B_frozen = nn.Parameter(
                frozen_B.detach().clone().float(),
                requires_grad=False,
            )
        else:
            self.A_frozen = None
            self.B_frozen = None

        # New trainable rank block
        if self.new_rank > 0:
            self.A_new = nn.Parameter(
                torch.zeros(self.new_rank, self.in_features)
            )

            self.B_new = nn.Parameter(
                torch.zeros(self.out_features, self.new_rank)
            )

            # Standard LoRA initialization
            nn.init.kaiming_uniform_(self.A_new, a=np.sqrt(5))
            nn.init.zeros_(self.B_new)

        else:
            # Final evaluation case:
            # all rank dimensions are frozen/copied.
            self.A_new = None
            self.B_new = None

    def full_A_B(self):
        """
        Return full A and B matrices:
        old frozen block + optional new trainable block.
        """

        A_parts = []
        B_parts = []

        if self.frozen_rank > 0:
            A_parts.append(
                self.A_frozen.to(
                    device=self.base_layer.weight.device,
                    dtype=self.base_layer.weight.dtype,
                )
            )

            B_parts.append(
                self.B_frozen.to(
                    device=self.base_layer.weight.device,
                    dtype=self.base_layer.weight.dtype,
                )
            )

        if self.new_rank > 0:
            A_parts.append(self.A_new)
            B_parts.append(self.B_new)

        if len(A_parts) == 0 or len(B_parts) == 0:
            raise ValueError("No LoRA rank blocks available.")

        A = torch.cat(A_parts, dim=0)
        B = torch.cat(B_parts, dim=1)

        return A, B

    def old_A_B(self):
        """
        Return old frozen A/B if available.
        Used for rank_extension_orth.
        """

        if self.frozen_rank <= 0:
            return None, None

        A_old = self.A_frozen.to(
            device=self.base_layer.weight.device,
            dtype=self.base_layer.weight.dtype,
        )

        B_old = self.B_frozen.to(
            device=self.base_layer.weight.device,
            dtype=self.base_layer.weight.dtype,
        )

        return A_old, B_old

    def new_A_B(self):
        """
        Return new trainable A/B if available.
        Used for rank_extension_orth.
        """

        if self.new_rank <= 0:
            return None, None

        return self.A_new, self.B_new

    def forward(self, x):
        """
        Forward:
            output = base_layer(x) + scaling * LoRA_update(x)

        LoRA update:
            x @ A.T @ B.T
        """

        base_out = self.base_layer(x)

        A, B = self.full_A_B()

        x_dropped = self.dropout(x)

        lora_hidden = torch.matmul(x_dropped, A.T)
        lora_out = torch.matmul(lora_hidden, B.T)

        return base_out + self.scaling * lora_out


def get_parent_module_and_child_name(model, module_name):
    """
    Given a module name like:
        vision_model.encoder.layers.0.self_attn.q_proj

    Return:
        parent module = vision_model.encoder.layers.0.self_attn
        child name    = q_proj
    """

    parts = module_name.split(".")
    parent = model

    for p in parts[:-1]:
        parent = getattr(parent, p)

    child_name = parts[-1]

    return parent, child_name


def find_clip_target_linear_modules(model):
    """
    Find q_proj and v_proj linear layers inside CLIP-ViT.
    """

    target_names = []

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            if name.endswith("q_proj") or name.endswith("v_proj"):
                target_names.append(name)

    return target_names


def build_rank_extension_model(previous_rank_state=None, step_idx=0):
    """
    Build CLIP-ViT model with growing-rank LoRA modules.

    If previous_rank_state is None:
        Step 1, rank = RANKEXT_NEW_RANK_PER_STEP

    Else:
        Copy previous A/B into frozen rank block and add new trainable rank block.
    """

    model = fresh_pretrained_model()

    # Freeze CLIP vision parameters
    for _, p in model.vision_model.named_parameters():
        p.requires_grad = False

    # Classifier remains trainable, but row gradients are masked per task
    for p in model.classifier.parameters():
        p.requires_grad = True

    total_rank = RANKEXT_NEW_RANK_PER_STEP * (step_idx + 1)

    target_names = find_clip_target_linear_modules(model)

    print(f"[rank_extension] Step {step_idx + 1}")
    print(f"  total_rank: {total_rank}")
    print(f"  target linear modules: {len(target_names)}")

    for module_name in target_names:
        parent, child_name = get_parent_module_and_child_name(model, module_name)
        base_layer = getattr(parent, child_name)

        frozen_A = None
        frozen_B = None

        if previous_rank_state is not None:
            if module_name in previous_rank_state["lora"]:
                frozen_A = previous_rank_state["lora"][module_name]["A"]
                frozen_B = previous_rank_state["lora"][module_name]["B"]

        growing_lora_layer = GrowingRankLoRALinear(
            base_layer=base_layer,
            total_rank=total_rank,
            frozen_A=frozen_A,
            frozen_B=frozen_B,
            dropout=LORA_DROPOUT,
        )

        setattr(parent, child_name, growing_lora_layer)

    # Restore previous classifier if available
    if previous_rank_state is not None:
        if previous_rank_state["classifier_weight"] is not None:
            with torch.no_grad():
                model.classifier.weight.copy_(
                    previous_rank_state["classifier_weight"].to(
                        device=model.classifier.weight.device,
                        dtype=model.classifier.weight.dtype,
                    )
                )

                model.classifier.bias.copy_(
                    previous_rank_state["classifier_bias"].to(
                        device=model.classifier.bias.device,
                        dtype=model.classifier.bias.dtype,
                    )
                )

    return model


def extract_rank_extension_state(model):
    """
    Extract full A/B matrices from all growing-rank LoRA layers,
    plus classifier weights.
    """

    state = {
        "lora": {},
        "classifier_weight": None,
        "classifier_bias": None,
    }

    for name, module in model.named_modules():
        if isinstance(module, GrowingRankLoRALinear):
            A, B = module.full_A_B()

            state["lora"][name] = {
                "A": A.detach().cpu().clone(),
                "B": B.detach().cpu().clone(),
                "scaling": float(module.scaling),
                "total_rank": int(module.total_rank),
                "frozen_rank": int(module.frozen_rank),
                "new_rank": int(module.new_rank),
                "rankext_alpha": float(module.rankext_alpha),
            }

    state["classifier_weight"] = model.classifier.weight.detach().cpu().clone()
    state["classifier_bias"] = model.classifier.bias.detach().cpu().clone()

    return state


def add_classifier_row_gradient_mask(model, current_classes):
    """
    Only allow classifier rows of the current classes to receive gradients.

    This is used in rank_extension so that old classifier rows are not overwritten.
    The masks must be created on the same device as the classifier parameters.
    """

    current_classes = set(int(c) for c in current_classes)

    weight_device = model.classifier.weight.device
    weight_dtype = model.classifier.weight.dtype

    bias_device = model.classifier.bias.device
    bias_dtype = model.classifier.bias.dtype

    mask_w = torch.zeros_like(
        model.classifier.weight,
        device=weight_device,
        dtype=weight_dtype,
    )

    mask_b = torch.zeros_like(
        model.classifier.bias,
        device=bias_device,
        dtype=bias_dtype,
    )

    for c in current_classes:
        mask_w[c, :] = 1.0
        mask_b[c] = 1.0

    hook_w = model.classifier.weight.register_hook(
        lambda grad: grad * mask_w.to(device=grad.device, dtype=grad.dtype)
    )

    hook_b = model.classifier.bias.register_hook(
        lambda grad: grad * mask_b.to(device=grad.device, dtype=grad.dtype)
    )

    return [hook_w, hook_b]


def compute_rankext_orth_penalty(model, eps=1e-8):
    """
    Orthogonal regularization for rank_extension_orth.

    The new trainable rank block is encouraged to be less aligned with
    the old frozen rank block inside each growing-rank LoRA layer.

    Penalty:
        mean cosine(delta_new, delta_old)^2

    where:
        delta_old = B_old @ A_old
        delta_new = B_new @ A_new
    """

    penalties = []

    for _, module in model.named_modules():
        if not isinstance(module, GrowingRankLoRALinear):
            continue

        A_old, B_old = module.old_A_B()
        A_new, B_new = module.new_A_B()

        if A_old is None or B_old is None:
            continue

        if A_new is None or B_new is None:
            continue

        old_delta = (B_old @ A_old) * float(module.scaling)
        new_delta = (B_new @ A_new) * float(module.scaling)

        old_vec = old_delta.flatten()
        new_vec = new_delta.flatten()

        old_norm = torch.linalg.norm(old_vec).clamp_min(eps)
        new_norm = torch.linalg.norm(new_vec).clamp_min(eps)

        cosine = torch.dot(old_vec, new_vec) / (old_norm * new_norm)

        penalties.append(cosine.pow(2))

    if len(penalties) == 0:
        return torch.tensor(0.0, device=next(model.parameters()).device)

    return torch.stack(penalties).mean()


class RankExtensionOrthTrainer(Trainer):
    """
    Trainer for rank_extension_orth.

    Loss:
        CE loss + lambda * orthogonal penalty

    This is not a standalone orthogonal method.
    It is rank_extension with an additional orthogonal regularization term.
    """

    def __init__(
        self,
        *args,
        lambda_orth=1e-3,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.lambda_orth = float(lambda_orth)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        outputs = model(**inputs)

        ce_loss = outputs.loss

        orth_loss = compute_rankext_orth_penalty(model)

        loss = ce_loss + self.lambda_orth * orth_loss

        return (loss, outputs) if return_outputs else loss


def print_rank_extension_trainable_parameters(model):
    """
    Print trainable parameter statistics for rank_extension.
    """

    trainable = 0
    total = 0

    for _, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()

    pct = 100.0 * trainable / total

    print(
        f"[rank_extension] trainable params: {trainable:,} || "
        f"all params: {total:,} || trainable%: {pct:.4f}"
    )


def run_rank_extension_variant(method_name, use_orth=False):
    """
    Run either:
        rank_extension
    or:
        rank_extension_orth

    Both use the same growing-rank LoRA architecture.
    The orth version adds an orthogonal regularization term between
    old frozen rank blocks and the new trainable rank block.
    """

    previous_rank_state = None

    for step_idx in range(NUM_STEPS):
        current_classes = classes_for_step(step_idx)

        model = build_rank_extension_model(
            previous_rank_state=previous_rank_state,
            step_idx=step_idx,
        )

        print_rank_extension_trainable_parameters(model)

        hooks = add_classifier_row_gradient_mask(
            model=model,
            current_classes=current_classes,
        )

        train_ds = make_train_dataset(
            step_idx=step_idx,
            replay_per_class=0,
        )

        eval_ds = make_eval_dataset(current_classes)

        out_dir = os.path.join(
            MODELS_DIR,
            f"{method_name}_step_{step_idx + 1}",
        )

        total_rank = (step_idx + 1) * RANKEXT_NEW_RANK_PER_STEP
        alpha_t = RANKEXT_ALPHA_PER_RANK * total_rank

        print(
            f"\n===== {method_name} | "
            f"step {step_idx + 1}/{NUM_STEPS} | "
            f"rank={total_rank} | "
            f"alpha={alpha_t:.1f} | "
            f"scaling={RANKEXT_ALPHA_PER_RANK:.2f} | "
            f"use_orth={use_orth} ====="
        )

        if use_orth:
            train_with_trainer(
                model=model,
                train_ds=train_ds,
                eval_ds=eval_ds,
                output_dir=out_dir,
                epochs=RANKEXT_EPOCHS,
                lr=LR_RANKEXT,
                batch_size=BATCH_LORA,
                accum_steps=ACCUM_LORA,
                trainer_cls=RankExtensionOrthTrainer,
                lambda_orth=RANKEXT_ORTH_LAMBDA,
            )
        else:
            train_with_trainer(
                model=model,
                train_ds=train_ds,
                eval_ds=eval_ds,
                output_dir=out_dir,
                epochs=RANKEXT_EPOCHS,
                lr=LR_RANKEXT,
                batch_size=BATCH_LORA,
                accum_steps=ACCUM_LORA,
            )

        for h in hooks:
            h.remove()

        previous_rank_state = extract_rank_extension_state(model)

        del model
        cleanup()

    final_rank_model = build_rank_extension_model(
        previous_rank_state=previous_rank_state,
        step_idx=NUM_STEPS - 1,
    )

    evaluate_model(final_rank_model, method_name)

    del final_rank_model
    cleanup()


if METHODS_TO_RUN["rank_extension"]:
    run_rank_extension_variant(
        method_name="rank_extension",
        use_orth=False,
    )
else:
    print("Skipping rank_extension")


if METHODS_TO_RUN["rank_extension_orth"]:
    run_rank_extension_variant(
        method_name="rank_extension_orth",
        use_orth=True,
    )
else:
    print("Skipping rank_extension_orth")

In [ ]:
# ============================================================
# 14) joint_upper_bound
# ============================================================

if METHODS_TO_RUN["joint_upper_bound"]:
    joint_model = fresh_pretrained_model()

    train_joint = make_joint_train_dataset()
    test_joint = make_joint_eval_dataset()

    print("\n===== joint_upper_bound =====")

    train_with_trainer(
        model=joint_model,
        train_ds=train_joint,
        eval_ds=test_joint,
        output_dir=os.path.join(MODELS_DIR, "joint_upper_bound"),
        epochs=JOINT_EPOCHS,
        lr=LR_JOINT,
        batch_size=BATCH_FT,
        accum_steps=ACCUM_FT,
    )

    evaluate_model(joint_model, "joint_upper_bound")

    del joint_model
    cleanup()

else:
    print("Skipping joint_upper_bound")

In [ ]:
# ============================================================
# 15) Final results table
# ============================================================

results_df = pd.DataFrame(all_results)

results_path = os.path.join(TABLES_DIR, "all_results_clip_vit_debug.csv")
results_df.to_csv(results_path, index=False)

print("Saved:", results_path)
results_df

In [ ]:
# ============================================================
# 18) Final pivot table + summary metrics
# ============================================================

method_order = [
    "seq_ft_no_replay",
    "simple_avg_no_replay",
    "simple_avg_replay",
    "simple_avg_replay_orth",
    "do_merging_simple",
    "do_merging_datafree_orth",
    "rank_extension",
    "rank_extension_orth",
    "joint_upper_bound",
]

results_df = pd.DataFrame(all_results)

results_path = os.path.join(TABLES_DIR, "all_results_clip_vit_full_comparison.csv")
results_df.to_csv(results_path, index=False)

print("Saved raw results:", results_path)
display(results_df)

final_table = results_df.pivot_table(
    index="method",
    columns="eval_set",
    values="accuracy",
    aggfunc="mean",
)

for col in ["first_step", "later_steps", "all_seen"]:
    if col not in final_table.columns:
        final_table[col] = np.nan

final_table = final_table.reindex(
    [m for m in method_order if m in final_table.index]
)

final_table = final_table[
    ["first_step", "later_steps", "all_seen"]
]

final_table_percent = final_table * 100

summary_table = final_table_percent.copy()

summary_table["old_new_gap"] = (
    summary_table["first_step"] - summary_table["later_steps"]
)

summary_table["min_old_new"] = summary_table[
    ["first_step", "later_steps"]
].min(axis=1)

summary_table["avg_old_new"] = summary_table[
    ["first_step", "later_steps"]
].mean(axis=1)

if "joint_upper_bound" in summary_table.index:
    joint_all_seen = summary_table.loc["joint_upper_bound", "all_seen"]

    summary_table["gap_to_joint_all_seen"] = (
        joint_all_seen - summary_table["all_seen"]
    )
else:
    summary_table["gap_to_joint_all_seen"] = np.nan

if (
    "simple_avg_no_replay" in summary_table.index
    and "simple_avg_replay" in summary_table.index
):
    replay_gain = (
        summary_table.loc["simple_avg_replay", ["first_step", "later_steps", "all_seen"]]
        - summary_table.loc["simple_avg_no_replay", ["first_step", "later_steps", "all_seen"]]
    )

    print("\nReplay gain: simple_avg_replay - simple_avg_no_replay")
    print(replay_gain.round(2))

if (
    "simple_avg_replay" in summary_table.index
    and "simple_avg_replay_orth" in summary_table.index
):
    replay_orth_gain = (
        summary_table.loc["simple_avg_replay_orth", ["first_step", "later_steps", "all_seen"]]
        - summary_table.loc["simple_avg_replay", ["first_step", "later_steps", "all_seen"]]
    )

    print("\nOrth gain on replay: simple_avg_replay_orth - simple_avg_replay")
    print(replay_orth_gain.round(2))

if (
    "do_merging_simple" in summary_table.index
    and "do_merging_datafree_orth" in summary_table.index
):
    do_gain = (
        summary_table.loc["do_merging_datafree_orth", ["first_step", "later_steps", "all_seen"]]
        - summary_table.loc["do_merging_simple", ["first_step", "later_steps", "all_seen"]]
    )

    print("\nDO data-free orth gain: do_merging_datafree_orth - do_merging_simple")
    print(do_gain.round(2))

if (
    "rank_extension" in summary_table.index
    and "rank_extension_orth" in summary_table.index
):
    rankext_orth_gain = (
        summary_table.loc["rank_extension_orth", ["first_step", "later_steps", "all_seen"]]
        - summary_table.loc["rank_extension", ["first_step", "later_steps", "all_seen"]]
    )

    print("\nOrth gain on rank_extension: rank_extension_orth - rank_extension")
    print(rankext_orth_gain.round(2))

final_table_path = os.path.join(TABLES_DIR, "final_accuracy_clip_vit_percent.csv")
summary_table_path = os.path.join(TABLES_DIR, "summary_metrics_clip_vit_percent.csv")

final_table_percent.to_csv(final_table_path)
summary_table.to_csv(summary_table_path)

print("\nSaved final accuracy table:", final_table_path)
print("Saved summary metrics table:", summary_table_path)

print("\nFinal accuracy (%):")
display(final_table_percent.round(2))

print("\nSummary metrics (%):")
display(summary_table.round(2))

In [ ]:
# ============================================================
# 19) More plots for comparison
# ============================================================

plot_df = final_table_percent.reset_index()

# ------------------------------------------------------------
# Plot 1: grouped bar accuracy
# ------------------------------------------------------------
x = np.arange(len(plot_df))
width = 0.25

plt.figure(figsize=(13, 6))

plt.bar(x - width, plot_df["first_step"], width, label="first_step")
plt.bar(x, plot_df["later_steps"], width, label="later_steps")
plt.bar(x + width, plot_df["all_seen"], width, label="all_seen")

plt.xticks(x, plot_df["method"], rotation=30, ha="right")
plt.ylabel("Accuracy (%)")
plt.title("CLIP-ViT + Split CIFAR-100: Final Accuracy Comparison")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "01_grouped_accuracy_comparison.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 2: heatmap accuracy
# ------------------------------------------------------------
plt.figure(figsize=(9, 5))

heatmap_data = final_table_percent[
    ["first_step", "later_steps", "all_seen"]
].copy()

plt.imshow(heatmap_data.values, aspect="auto")
plt.colorbar(label="Accuracy (%)")

plt.xticks(
    ticks=np.arange(len(heatmap_data.columns)),
    labels=heatmap_data.columns,
    rotation=0,
)

plt.yticks(
    ticks=np.arange(len(heatmap_data.index)),
    labels=heatmap_data.index,
)

for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        value = heatmap_data.iloc[i, j]
        if not np.isnan(value):
            plt.text(j, i, f"{value:.1f}", ha="center", va="center")

plt.title("Accuracy Heatmap (%)")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "02_accuracy_heatmap.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 3: all_seen ranking
# ------------------------------------------------------------
ranking_df = final_table_percent.sort_values("all_seen", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(ranking_df.index, ranking_df["all_seen"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("All-seen accuracy (%)")
plt.title("Method Ranking by All-seen Accuracy")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "03_all_seen_ranking.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 4: old-new gap
# ------------------------------------------------------------
gap_df = summary_table.copy()

plt.figure(figsize=(10, 5))
plt.bar(gap_df.index, gap_df["old_new_gap"])
plt.axhline(0, linestyle="--", linewidth=1)
plt.xticks(rotation=30, ha="right")
plt.ylabel("first_step - later_steps accuracy (%)")
plt.title("Old-New Accuracy Gap")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "04_old_new_gap.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 5: gap to joint upper bound
# ------------------------------------------------------------
if "gap_to_joint_all_seen" in summary_table.columns:
    gap_joint_df = summary_table.drop(index="joint_upper_bound", errors="ignore")

    plt.figure(figsize=(10, 5))
    plt.bar(gap_joint_df.index, gap_joint_df["gap_to_joint_all_seen"])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Gap to joint all-seen accuracy (%)")
    plt.title("Distance from Joint Upper Bound")
    plt.tight_layout()

    plot_path = os.path.join(PLOTS_DIR, "05_gap_to_joint_upper_bound.png")
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 6: replay gain
# ------------------------------------------------------------
if (
    "simple_avg_no_replay" in final_table_percent.index
    and "simple_avg_replay" in final_table_percent.index
):
    replay_gain_series = (
        final_table_percent.loc["simple_avg_replay", ["first_step", "later_steps", "all_seen"]]
        - final_table_percent.loc["simple_avg_no_replay", ["first_step", "later_steps", "all_seen"]]
    )

    plt.figure(figsize=(7, 5))
    plt.bar(replay_gain_series.index, replay_gain_series.values)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.ylabel("Accuracy gain (%)")
    plt.title("Replay Gain: simple_avg_replay - simple_avg_no_replay")
    plt.tight_layout()

    plot_path = os.path.join(PLOTS_DIR, "06_replay_gain.png")
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print("Saved:", plot_path)


# ------------------------------------------------------------
# Plot 7: loss comparison
# ------------------------------------------------------------
loss_table = results_df.pivot_table(
    index="method",
    columns="eval_set",
    values="loss",
    aggfunc="mean",
)

loss_table = loss_table.reindex(
    [m for m in method_order if m in loss_table.index]
)

for col in ["first_step", "later_steps", "all_seen"]:
    if col not in loss_table.columns:
        loss_table[col] = np.nan

loss_table = loss_table[
    ["first_step", "later_steps", "all_seen"]
]

loss_table_path = os.path.join(TABLES_DIR, "final_loss_table.csv")
loss_table.to_csv(loss_table_path)

loss_plot_df = loss_table.reset_index()

x = np.arange(len(loss_plot_df))
width = 0.25

plt.figure(figsize=(13, 6))

plt.bar(x - width, loss_plot_df["first_step"], width, label="first_step")
plt.bar(x, loss_plot_df["later_steps"], width, label="later_steps")
plt.bar(x + width, loss_plot_df["all_seen"], width, label="all_seen")

plt.xticks(x, loss_plot_df["method"], rotation=30, ha="right")
plt.ylabel("Loss")
plt.title("Evaluation Loss Comparison")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "07_loss_comparison.png")
plt.savefig(plot_path, dpi=200)
plt.show()

print("Saved loss table:", loss_table_path)
print("Saved:", plot_path)

In [ ]:
# ============================================================
# 20) Explanation for supervisor
# ============================================================

print(f"""
Supervisor-aligned full comparison setup:

Backbone:
- CLIP-ViT vision encoder: {MODEL_CHECKPOINT}
- Text encoder is not used.
- A new CIFAR-100 classifier is trained on top of the CLIP vision representation.
- LoRA target modules are q_proj and v_proj.

Continual learning setup:
- Split CIFAR-100
- 5 steps
- 20 classes per step
- Evaluation sets:
  1. first_step   = classes 0-19
  2. later_steps  = classes 20-99
  3. all_seen     = classes 0-99

Compared methods:

1. seq_ft_no_replay
   Sequential full fine-tuning without replay.
   This is the catastrophic-forgetting baseline.

2. simple_avg_no_replay
   One independent LoRA trained per step without replay.
   The LoRA deltas are merged using simple average.
   This is the main no-replay merging method.

3. simple_avg_replay
   Same as simple_avg_no_replay, but each step uses a small replay buffer.
   Replay is used as a diagnostic to measure the effect of old-data exposure.

4. simple_avg_replay_orth
   Same as simple_avg_replay, but with orthogonal regularization during LoRA training.
   The regularizer encourages the current LoRA delta to be less aligned with previous LoRA deltas.

5. do_merging_simple
   Simplified DO-inspired variant.
   It globally decouples LoRA deltas into magnitude and direction before merging.
   This is not the full DO-Merging-style variant.

6. do_merging_datafree_orth
   Closer DO-Merging-style implementation.
   It performs data-free orthogonalization of LoRA A/B components across tasks,
   then builds full deltas, decouples them column-wise into magnitude and direction,
   and merges magnitude and direction separately.

7. rank_extension
   Pietro/Leon rank-extension variant.
   One LoRA grows in rank step by step.
   Old rank parts are copied and frozen; only the new rank block is trained.

8. rank_extension_orth
   Same as rank_extension, but with orthogonal regularization.
   The new trainable rank block is encouraged to be less aligned with the old frozen rank block.

9. joint_upper_bound
   Joint training on all classes.
   This is not a continual-learning method; it is the upper-bound reference.

Epochs:
- FT_EPOCHS = {FT_EPOCHS}
- LORA_EPOCHS = {LORA_EPOCHS}
- JOINT_EPOCHS = {JOINT_EPOCHS}
- RANKEXT_EPOCHS = {RANKEXT_EPOCHS}

DO-Merging-style settings:
- DO_ORTH_STEPS = {DO_ORTH_STEPS}
- DO_ORTH_LR = {DO_ORTH_LR}
- DO_PRESERVE_WEIGHT = {DO_PRESERVE_WEIGHT}
- DO_ORTH_WEIGHT = {DO_ORTH_WEIGHT}

Main interpretation to check:
- Does simple_avg_no_replay beat seq_ft_no_replay?
- Does simple_avg_replay improve over simple_avg_no_replay?
- Does simple_avg_replay_orth improve over simple_avg_replay?
- Does do_merging_datafree_orth improve over do_merging_simple and simple_avg_no_replay?
- Does rank_extension_orth improve over rank_extension?
- How far are all methods from joint_upper_bound?
""")